# Bayes' theorem, one word at a time

A spam filter has to answer one question for every message: given the words in it, how likely is it to be spam? The words are the evidence, and spam or not is the hypothesis. Bayes' theorem is the rule for turning one into the other. Here it is applied to a single word, by counting, on a real collection of text messages.

## The messages

The SMS Spam Collection: 5,574 text messages in English, each labelled **spam** or **ham** (a legitimate message). It was put together for research on spam filters (Almeida, Gómez Hidalgo and Yamakami, 2011) and is published by the UCI Machine Learning Repository under a CC BY 4.0 licence. `load_sms_spam`, in this folder's `sms_spam.py`, downloads it the first time and reads it from disk afterwards:

In [ ]:
from sms_spam import load_sms_spam

messages = load_sms_spam()
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
messages["label"].value_counts()

label
ham     4827
spam     747
Name: count, dtype: int64

To ask whether a message contains a word, each message is turned into the set of its words: lower case, split wherever there is something other than a letter, a digit or `£`, which spam uses for its prizes.

In [ ]:
import re

def words_in(message):
    return set(re.findall(r"[a-z0-9£]+", message.lower()))

messages["words"] = messages["message"].map(words_in)
messages.loc[2, ["message", "words"]].to_dict()

{'message': "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's",
 'words': {'08452810075over18',
  '2',
  '2005',
  '21st',
  '87121',
  'a',
  'apply',
  'c',
  'comp',
  'cup',
  'entry',
  'fa',
  'final',
  'free',
  'in',
  'may',
  'question',
  'rate',
  'receive',
  's',
  'std',
  't',
  'text',
  'tkts',
  'to',
  'txt',
  'win',
  'wkly'}}

## The prior

Before reading a message, the best estimate of the probability that it is spam is the share of spam in the collection. This is the **prior** probability, $P(\text{spam})$:

In [ ]:
is_spam = messages["label"] == "spam"
prior_spam = is_spam.mean()
prior_spam

np.float64(0.1340150699677072)

## One word as evidence

Now a word appears: "free". Two numbers describe how it behaves in each class: the share of spam messages that contain it, $P(\text{free} \mid \text{spam})$, and the share of ham messages that contain it, $P(\text{free} \mid \text{ham})$. They are called **likelihoods**:

In [ ]:
contains_free = messages["words"].map(lambda words: "free" in words)
likelihood_spam = contains_free[is_spam].mean()
likelihood_ham = contains_free[~is_spam].mean()
likelihood_spam, likelihood_ham

(np.float64(0.22757697456492637), np.float64(0.012222912782266417))

About 23% of the spam messages contain "free", against about 1.2% of the ham messages: the word is almost 19 times more frequent in spam. That alone doesn't say how likely a message with "free" is to be spam, because that also depends on how much spam there is to begin with.

**Bayes' theorem** combines the prior and the likelihoods into the **posterior**, the probability of spam once the word is known:

$$P(\text{spam} \mid \text{free}) = \frac{P(\text{free} \mid \text{spam}) \, P(\text{spam})}{P(\text{free})}$$

where $P(\text{free})$, the share of all messages that contain the word, comes from both classes:

$$P(\text{free}) = P(\text{free} \mid \text{spam}) \, P(\text{spam}) + P(\text{free} \mid \text{ham}) \, P(\text{ham})$$

In [ ]:
evidence = likelihood_spam * prior_spam + likelihood_ham * (1 - prior_spam)
posterior_spam = likelihood_spam * prior_spam / evidence
posterior_spam

np.float64(0.7423580786026202)

The same number can be counted directly: of the messages that contain "free", the share that are spam:

In [ ]:
is_spam[contains_free].mean()

np.float64(0.74235807860262)

0.742 both ways: 74% of the messages that contain "free" are spam. On counts, Bayes' theorem is not an approximation; it gives exactly what direct counting gives. Its value is in its parts: the likelihoods describe the words, and the prior describes the mix of messages, so the prior can change without counting again (below), and the likelihoods of several words can be combined, which [A naive Bayes classifier by hand](naive_bayes_by_hand.ipynb) does. A probability counted directly for one word can't be combined with another's.

## Many words, one at a time

The same calculation for a few words, each on its own:

In [ ]:
import pandas as pd


def bayes_for_word(word):
    contains = messages["words"].map(lambda words: word in words)
    likelihood_spam = contains[is_spam].mean()
    likelihood_ham = contains[~is_spam].mean()
    evidence = likelihood_spam * prior_spam + likelihood_ham * (1 - prior_spam)
    return {
        "messages": contains.sum(),
        "P(word | spam)": likelihood_spam,
        "P(word | ham)": likelihood_ham,
        "P(spam | word)": likelihood_spam * prior_spam / evidence,
    }


pd.DataFrame({word: bayes_for_word(word) for word in ["txt", "free", "claim", "call", "now", "you", "ok", "love", "lol"]}).T.round(3)

,messages,P(word | spam),P(word | ham),P(spam | word)
txt,165.0,0.203,0.003,0.921
free,229.0,0.228,0.012,0.742
claim,108.0,0.145,0.000,1.000
call,551.0,0.439,0.046,0.595
now,484.0,0.258,0.060,0.399
you,1591.0,0.324,0.279,0.152
ok,280.0,0.007,0.057,0.018
love,178.0,0.011,0.035,0.045
lol,74.0,0.000,0.015,0.000


Some words point to spam ("txt", "free", "claim"), some to ham ("ok", "love", "lol"), and some, like "you", barely move the probability from the prior of 0.134, because they are common in both classes. Two rows deserve suspicion: "claim" gives 1.000 and "lol" 0.000, certainty from a single word, only because "claim" never appears in these ham messages and "lol" never in this spam. A filter that trusted them would send to the spam folder any message containing "claim", whatever else it said. [A naive Bayes classifier by hand](naive_bayes_by_hand.ipynb) fixes this, with *smoothing*.

## How much the prior matters

The posterior depends on the prior as much as on the word. Suppose the same filter worked on a phone that receives as much spam as ham, with the same likelihoods for "free":

In [ ]:
def posterior(likelihood_spam, likelihood_ham, prior_spam):
    return likelihood_spam * prior_spam / (likelihood_spam * prior_spam + likelihood_ham * (1 - prior_spam))


pd.Series({prior: posterior(likelihood_spam, likelihood_ham, prior) for prior in (0.01, 0.05, 0.134, 0.3, 0.5)}).rename_axis("prior P(spam)").rename("P(spam | free)").round(3)

prior P(spam)
0.010    0.158
0.050    0.495
0.134    0.742
0.300    0.889
0.500    0.949
Name: P(spam | free), dtype: float64

With 1% of spam, the same word gives only a 16% probability of spam; with half the messages spam, 95%. The evidence of a word always acts on how likely the hypothesis was before seeing it. Forgetting the prior is the classic mistake when reading a medical test, and the same mistake in a spam filter.

The theorem itself, its derivation from conditional probability, and the same reasoning applied to a medical test, where the prior is the share of people who have a disease, are in [Bayes' theorem](../../../concepts/deep-dives/probability/bayes_theorem.md) and [a diagnostic test](../../../concepts/deep-dives/probability/diagnostic_test.md).

One word is weak evidence. The next notebook, [A naive Bayes classifier by hand](naive_bayes_by_hand.ipynb), combines every word of a message into one probability.